# Module 6 - Class 5: Sequence Classification with LSTMs
**Khamidullokhon Abduvokhidov**

In [ ]:
# Load IMDB reviews and pad every review to 200 word indices.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time, tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Bidirectional
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
(X_train,y_train),(X_test,y_test)=imdb.load_data(num_words=10000)
X_train=pad_sequences(X_train,maxlen=200); X_test=pad_sequences(X_test,maxlen=200)

In [ ]:
# Train a logistic-regression baseline on padded word-index features.
start=time.time(); lr=LogisticRegression(max_iter=1000).fit(X_train,y_train); lr_time=time.time()-start; lr_acc=accuracy_score(y_test,lr.predict(X_test))
print('LogReg accuracy:',lr_acc,'time:',lr_time)

In [ ]:
# Train unidirectional and bidirectional LSTM models on the same reviews.
def build(bi=False):
  layer=Bidirectional(LSTM(64)) if bi else LSTM(64)
  m=Sequential([Embedding(10000,128),layer,Dense(1,activation='sigmoid')]); m.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy']); return m
lstm=build(); start=time.time(); h_lstm=lstm.fit(X_train,y_train,epochs=5,batch_size=64,validation_split=.2); lstm_time=time.time()-start; _,lstm_acc=lstm.evaluate(X_test,y_test)
bilstm=build(True); start=time.time(); h_bi=bilstm.fit(X_train,y_train,epochs=5,batch_size=64,validation_split=.2); bi_time=time.time()-start; _,bi_acc=bilstm.evaluate(X_test,y_test)
display(pd.DataFrame([['LogReg',lr_acc,lr_time,lr.count_params()],['LSTM',lstm_acc,lstm_time,lstm.count_params()],['BiLSTM',bi_acc,bi_time,bilstm.count_params()]],columns=['Model','Test Accuracy','Training Time','Parameters']))
plt.plot(h_lstm.history['val_accuracy'],label='LSTM'); plt.plot(h_bi.history['val_accuracy'],label='BiLSTM'); plt.legend(); plt.title('Validation Accuracy'); plt.show()

## Analysis
The LSTM has an advantage when word order and longer contextual patterns matter, because its embedding and recurrent layers learn sequence representations. Logistic regression is simpler, faster, and can be preferable when its accuracy is close to the deep model. The table above gives the tradeoff for this run; the added complexity is justified only when its validation or test gain matters for the application.